<a href="https://colab.research.google.com/github/EngrAsadKhan/Assignment-PAI-/blob/main/Neuro_Symbolic_Fuzzy_008.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
STEP 1: MOUNT DRIVE, EXTRACT DATASET, AND EXPLORE

INSTRUCTIONS:
1. Go to https://colab.research.google.com
2. Create new notebook
3. Copy this ENTIRE code into ONE cell
4. Update ZIP_FILE_PATH below with your actual path
5. Run the cell (Shift + Enter)
"""

# ============================================
# CONFIGURATION - CHANGE THIS PATH
# ============================================

# PASTE YOUR ZIP FILE PATH HERE
ZIP_FILE_PATH = "/content/drive/MyDrive/indiana dataset.zip"  # ← CHANGE THIS TO YOUR PATH

# ============================================
# STEP 1: MOUNT GOOGLE DRIVE
# ============================================

from google.colab import drive
import os
import zipfile
import shutil
import pandas as pd

print("="*70)
print("STEP 1: MOUNTING GOOGLE DRIVE")
print("="*70)

drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!\n")

# ============================================
# STEP 2: VERIFY ZIP FILE EXISTS
# ============================================

print("="*70)
print("STEP 2: CHECKING ZIP FILE")
print("="*70)

print(f"Looking for: {ZIP_FILE_PATH}")

if os.path.exists(ZIP_FILE_PATH):
    file_size_mb = os.path.getsize(ZIP_FILE_PATH) / (1024 * 1024)
    print(f"✓ ZIP file found!")
    print(f"✓ File size: {file_size_mb:.2f} MB\n")
else:
    print("✗ ZIP file NOT found!")
    print("\nPlease update ZIP_FILE_PATH variable at the top of this code")
    print("Example: '/content/drive/MyDrive/your_folder/your_file.zip'")
    raise FileNotFoundError(f"ZIP file not found at: {ZIP_FILE_PATH}")

# ============================================
# STEP 3: EXTRACT DATASET
# ============================================

print("="*70)
print("STEP 3: EXTRACTING DATASET")
print("="*70)

EXTRACT_PATH = "/content/indiana_dataset"

# Remove old extraction if exists
if os.path.exists(EXTRACT_PATH):
    print("Removing previous extraction...")
    shutil.rmtree(EXTRACT_PATH)

print(f"Extracting to: {EXTRACT_PATH}")
print("Please wait...")

try:
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        # Get list of files
        file_list = zip_ref.namelist()
        print(f"Total files in ZIP: {len(file_list)}")

        # Extract
        zip_ref.extractall(EXTRACT_PATH)

    print("✓ Extraction complete!\n")
except Exception as e:
    print(f"✗ Extraction failed: {e}")
    raise

# ============================================
# STEP 4: EXPLORE DIRECTORY STRUCTURE
# ============================================

print("="*70)
print("STEP 4: EXPLORING DATASET STRUCTURE")
print("="*70)

def show_directory_tree(path, prefix="", max_files=5):
    """Show directory structure"""
    try:
        items = sorted(os.listdir(path))

        folders = [item for item in items if os.path.isdir(os.path.join(path, item))]
        files = [item for item in items if os.path.isfile(os.path.join(path, item))]

        # Show folders
        for i, folder in enumerate(folders):
            is_last_folder = (i == len(folders) - 1) and len(files) == 0
            connector = "└── " if is_last_folder else "├── "

            folder_path = os.path.join(path, folder)
            num_items = len(os.listdir(folder_path))
            print(f"{prefix}{connector}📁 {folder}/ ({num_items} items)")

            # Recursively show subfolder (1 level only)
            new_prefix = prefix + ("    " if is_last_folder else "│   ")
            sub_items = os.listdir(folder_path)
            if sub_items:
                for j, sub_item in enumerate(sorted(sub_items)[:3]):
                    sub_connector = "└── " if j == min(2, len(sub_items)-1) else "├── "
                    print(f"{new_prefix}{sub_connector}{sub_item}")
                if len(sub_items) > 3:
                    print(f"{new_prefix}    ... and {len(sub_items)-3} more")

        # Show files
        for i, file in enumerate(files[:max_files]):
            is_last = i == min(len(files), max_files) - 1
            connector = "└── " if is_last else "├── "

            file_path = os.path.join(path, file)
            size_kb = os.path.getsize(file_path) / 1024
            if size_kb > 1024:
                size_str = f"{size_kb/1024:.2f} MB"
            else:
                size_str = f"{size_kb:.2f} KB"

            print(f"{prefix}{connector}📄 {file} ({size_str})")

        if len(files) > max_files:
            print(f"{prefix}    ... and {len(files)-max_files} more files")

    except Exception as e:
        print(f"{prefix}✗ Error: {e}")

print("\nDataset structure:")
print(f"📁 {EXTRACT_PATH}/")
show_directory_tree(EXTRACT_PATH)

# ============================================
# STEP 5: LOCATE KEY COMPONENTS
# ============================================

print("\n" + "="*70)
print("STEP 5: LOCATING DATASET COMPONENTS")
print("="*70)

# Find images folder
images_folder = None
image_count = 0

print("\n🔍 Searching for images folder...")
for root, dirs, files in os.walk(EXTRACT_PATH):
    for dirname in dirs:
        if dirname.lower() in ['images', 'png', 'jpg', 'pictures', 'pics']:
            folder_path = os.path.join(root, dirname)
            # Check if it contains image files
            imgs = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.dcm'))]
            if imgs:
                images_folder = folder_path
                image_count = len(imgs)
                break
    if images_folder:
        break

if images_folder:
    print(f"✓ Images folder found: {images_folder}")
    print(f"  Total images: {image_count}")

    # Show sample filenames
    sample_images = os.listdir(images_folder)[:5]
    print(f"  Sample filenames:")
    for img in sample_images:
        print(f"    - {img}")
else:
    print("⚠ Images folder not found automatically")
    print("  Will search for image files in all folders...")

# Find CSV files
print("\n🔍 Searching for CSV files...")
csv_files = []

for root, dirs, files in os.walk(EXTRACT_PATH):
    for file in files:
        if file.endswith('.csv'):
            csv_path = os.path.join(root, file)
            csv_files.append((file, csv_path))

if csv_files:
    print(f"✓ Found {len(csv_files)} CSV file(s):")
    for filename, filepath in csv_files:
        # Get file size
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  - {filename} ({size_kb:.2f} KB)")
        print(f"    Path: {filepath}")
else:
    print("✗ No CSV files found")

# ============================================
# STEP 6: PREVIEW CSV FILES
# ============================================

print("\n" + "="*70)
print("STEP 6: PREVIEWING CSV FILES")
print("="*70)

csv_dataframes = {}

for filename, filepath in csv_files:
    print(f"\n📊 FILE: {filename}")
    print("-" * 70)

    try:
        df = pd.read_csv(filepath)
        csv_dataframes[filename] = df

        print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"\n  Columns ({len(df.columns)}):")
        for i, col in enumerate(df.columns, 1):
            print(f"    {i}. {col}")

        print(f"\n  First 3 rows:")
        print(df.head(3).to_string(max_cols=10))

        # Check for missing values
        missing = df.isnull().sum()
        if missing.sum() > 0:
            print(f"\n  Missing values:")
            for col, count in missing[missing > 0].items():
                print(f"    - {col}: {count} ({count/len(df)*100:.1f}%)")

    except Exception as e:
        print(f"  ✗ Error reading CSV: {e}")

# ============================================
# STEP 7: SUMMARY AND NEXT STEPS
# ============================================

print("\n" + "="*70)
print("✓ STEP 1 COMPLETE - DATASET READY!")
print("="*70)

print("\n📦 DATASET SUMMARY:")
print(f"  Location: {EXTRACT_PATH}")
print(f"  Images: {image_count} files" if images_folder else "  Images: Not located")
print(f"  CSV files: {len(csv_files)}")

if images_folder:
    print(f"\n📁 Images folder: {images_folder}")

print(f"\n📋 CSV Files:")
for filename, filepath in csv_files:
    if filename in csv_dataframes:
        df = csv_dataframes[filename]
        print(f"  • {filename}: {len(df)} rows, {len(df.columns)} columns")

print("\n" + "="*70)
print("READY FOR STEP 2: DATA LOADING")
print("="*70)

print("\n💡 PLEASE PROVIDE THIS INFORMATION:")
print("  1. Column name that contains image filenames")
print("  2. Column name that contains radiology reports/findings")
print("  3. Are there any other important columns?")

print("\nOnce you provide this info, I'll create Step 2 code!")

# Save paths for next step
print("\n📌 PATHS TO USE IN NEXT STEP:")
print(f"DATASET_PATH = '{EXTRACT_PATH}'")
if images_folder:
    print(f"IMAGES_PATH = '{images_folder}'")
for filename, filepath in csv_files:
    var_name = filename.replace('.csv', '').replace('-', '_').replace(' ', '_').upper()
    print(f"{var_name}_CSV = '{filepath}'")

STEP 1: MOUNTING GOOGLE DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted successfully!

STEP 2: CHECKING ZIP FILE
Looking for: /content/drive/MyDrive/indiana dataset.zip
✓ ZIP file found!
✓ File size: 13479.72 MB

STEP 3: EXTRACTING DATASET
Removing previous extraction...
Extracting to: /content/indiana_dataset
Please wait...
Total files in ZIP: 7472


In [ ]:
"""
STEP 2: LOAD AND PROCESS INDIANA DATASET
- Merge projections and reports
- Filter frontal images only
- Extract CheXpert-style labels from reports
- Create train/test split

INSTRUCTIONS:
1. Copy this ENTIRE code into a NEW cell in your Colab notebook
2. Run it (Shift + Enter)
"""

import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

# ============================================
# PATHS FROM STEP 1
# ============================================

DATASET_PATH = '/content/indiana_dataset'
INDIANA_REPORTS_CSV = '/content/indiana_dataset/indiana_reports.csv'
INDIANA_PROJECTIONS_CSV = '/content/indiana_dataset/indiana_projections.csv'

# Find images folder
IMAGES_PATH = os.path.join(DATASET_PATH, 'images', 'images_normalized')

print("="*70)
print("STEP 2: LOADING AND PROCESSING INDIANA DATASET")
print("="*70)

# ============================================
# STEP 2.1: LOAD CSV FILES
# ============================================

print("\nStep 2.1: Loading CSV files...")

# Load reports
reports_df = pd.read_csv(INDIANA_REPORTS_CSV)
print(f"✓ Reports loaded: {len(reports_df)} records")
print(f"  Columns: {list(reports_df.columns)}")

# Load projections
projections_df = pd.read_csv(INDIANA_PROJECTIONS_CSV)
print(f"✓ Projections loaded: {len(projections_df)} records")
print(f"  Columns: {list(projections_df.columns)}")

# ============================================
# STEP 2.2: MERGE DATA
# ============================================

print("\n" + "="*70)
print("Step 2.2: Merging projections and reports...")
print("="*70)

# Merge on 'uid' column
merged_df = pd.merge(
    projections_df,
    reports_df,
    on='uid',
    how='inner'
)

print(f"✓ Merged dataset: {len(merged_df)} records")
print(f"  Total columns: {len(merged_df.columns)}")

# ============================================
# STEP 2.3: FILTER FRONTAL IMAGES ONLY
# ============================================

print("\n" + "="*70)
print("Step 2.3: Filtering frontal images only...")
print("="*70)

print(f"\nProjection types:")
print(projections_df['projection'].value_counts())

# Keep only Frontal images (remove Lateral)
frontal_df = merged_df[merged_df['projection'] == 'Frontal'].copy()
frontal_df = frontal_df.reset_index(drop=True)

print(f"\n✓ Filtered to frontal images: {len(frontal_df)} records")
print(f"  Removed lateral images: {len(merged_df) - len(frontal_df)}")

# ============================================
# STEP 2.4: VERIFY IMAGE FILES EXIST
# ============================================

print("\n" + "="*70)
print("Step 2.4: Verifying image files...")
print("="*70)

print(f"Images folder: {IMAGES_PATH}")

if os.path.exists(IMAGES_PATH):
    image_files = [f for f in os.listdir(IMAGES_PATH)
                   if f.endswith(('.png', '.jpg', '.jpeg', '.dcm'))]
    print(f"✓ Found {len(image_files)} image files")

    # Show sample filenames
    print(f"\nSample image filenames:")
    for img in image_files[:5]:
        print(f"  - {img}")

    # Check if filenames in CSV match files in folder
    sample_filename = frontal_df['filename'].iloc[0]
    full_path = os.path.join(IMAGES_PATH, sample_filename)

    if os.path.exists(full_path):
        print(f"\n✓ Image files match CSV filenames")
    else:
        print(f"\n⚠ Warning: Sample filename not found: {sample_filename}")
else:
    print(f"✗ Images folder not found at: {IMAGES_PATH}")
    print("Checking alternative locations...")

    # Try to find images
    for root, dirs, files in os.walk(DATASET_PATH):
        png_files = [f for f in files if f.endswith('.png')]
        if len(png_files) > 100:  # Found the images folder
            IMAGES_PATH = root
            print(f"✓ Found images at: {IMAGES_PATH}")
            break

# ============================================
# STEP 2.5: EXTRACT CHEXPERT LABELS FROM REPORTS
# ============================================

print("\n" + "="*70)
print("Step 2.5: Extracting CheXpert-style labels from reports...")
print("="*70)

# CheXpert label categories
CHEXPERT_LABELS = [
    'No Finding',
    'Enlarged Cardiomediastinum',
    'Cardiomegaly',
    'Lung Opacity',
    'Lung Lesion',
    'Edema',
    'Consolidation',
    'Pneumonia',
    'Atelectasis',
    'Pneumothorax',
    'Pleural Effusion',
    'Pleural Other',
    'Fracture',
    'Support Devices'
]

# Keywords for each finding
LABEL_KEYWORDS = {
    'Cardiomegaly': ['cardiomegaly', 'cardiac enlargement', 'enlarged heart', 'enlarged cardiac'],
    'Edema': ['edema', 'pulmonary edema', 'interstitial edema'],
    'Consolidation': ['consolidation', 'consolidative', 'airspace consolidation'],
    'Pneumonia': ['pneumonia', 'infiltrate', 'infection'],
    'Atelectasis': ['atelectasis', 'collapse', 'atelectatic'],
    'Pneumothorax': ['pneumothorax', 'ptx'],
    'Pleural Effusion': ['pleural effusion', 'effusion', 'pleural fluid'],
    'Lung Opacity': ['opacity', 'opacities', 'opacification', 'airspace disease'],
    'Lung Lesion': ['lesion', 'nodule', 'mass', 'density'],
    'Fracture': ['fracture', 'broken', 'fx'],
    'Enlarged Cardiomediastinum': ['widened mediastinum', 'mediastinal widening', 'wide mediastinum'],
    'Support Devices': ['tube', 'catheter', 'device', 'pacemaker', 'lead', 'wire', 'picc'],
    'Pleural Other': ['pleural thickening', 'pleural abnormality', 'pleural disease']
}

# Initialize label columns
for label in CHEXPERT_LABELS:
    frontal_df[label] = 0

# Extract labels
print("Extracting labels from findings and impressions...")

for idx, row in frontal_df.iterrows():
    # Combine findings and impression text
    findings_text = str(row.get('findings', '')).lower()
    impression_text = str(row.get('impression', '')).lower()
    combined_text = findings_text + ' ' + impression_text

    # Check for normal/clear chest
    normal_keywords = ['normal', 'clear', 'unremarkable', 'no acute']
    is_normal = any(keyword in combined_text for keyword in normal_keywords)

    # Check if any abnormality is mentioned
    has_finding = False

    # Check for each finding
    for label, keywords in LABEL_KEYWORDS.items():
        if any(keyword in combined_text for keyword in keywords):
            frontal_df.at[idx, label] = 1
            has_finding = True

    # If no findings and text suggests normal
    if not has_finding and is_normal:
        frontal_df.at[idx, 'No Finding'] = 1

# ============================================
# STEP 2.6: LABEL STATISTICS
# ============================================

print("\n" + "="*70)
print("Label Distribution:")
print("="*70)

label_stats = []
for label in CHEXPERT_LABELS:
    count = frontal_df[label].sum()
    percentage = (count / len(frontal_df)) * 100
    label_stats.append({'Label': label, 'Count': count, 'Percentage': percentage})
    print(f"  {label:30s}: {count:4d} ({percentage:5.2f}%)")

label_stats_df = pd.DataFrame(label_stats)

# Plot distribution
plt.figure(figsize=(12, 6))
plt.barh(label_stats_df['Label'], label_stats_df['Count'], color='steelblue')
plt.xlabel('Number of Cases')
plt.title('CheXpert Label Distribution in Indiana Dataset')
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Label distribution saved as 'label_distribution.png'")

# ============================================
# STEP 2.7: CREATE TRAIN/TEST SPLIT
# ============================================

print("\n" + "="*70)
print("Step 2.7: Creating train/test split...")
print("="*70)

# Split 80% train, 20% test
train_df, test_df = train_test_split(
    frontal_df,
    test_size=0.2,
    random_state=42,
    stratify=frontal_df['No Finding']  # Stratify by normal/abnormal
)

print(f"✓ Train set: {len(train_df)} images ({len(train_df)/len(frontal_df)*100:.1f}%)")
print(f"✓ Test set: {len(test_df)} images ({len(test_df)/len(frontal_df)*100:.1f}%)")

# Show label distribution in splits
print("\nLabel distribution in train/test:")
print("-" * 70)
print(f"{'Label':<30s} {'Train':>8s} {'Test':>8s}")
print("-" * 70)

for label in CHEXPERT_LABELS:
    train_count = train_df[label].sum()
    test_count = test_df[label].sum()
    print(f"{label:<30s} {train_count:>8d} {test_count:>8d}")

# ============================================
# STEP 2.8: VISUALIZE SAMPLE IMAGES
# ============================================

print("\n" + "="*70)
print("Step 2.8: Visualizing sample images...")
print("="*70)

def load_and_display_image(filename, title=""):
    """Load and display an image"""
    img_path = os.path.join(IMAGES_PATH, filename)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        return np.array(img), title
    else:
        return None, f"Image not found: {filename}"

# Visualize 6 random samples
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

samples = train_df.sample(n=6, random_state=42)

for idx, (ax, (_, row)) in enumerate(zip(axes, samples.iterrows())):
    filename = row['filename']
    img, title = load_and_display_image(filename)

    if img is not None:
        ax.imshow(img, cmap='gray')
        ax.axis('off')

        # Get positive findings
        positive_labels = [label for label in CHEXPERT_LABELS if row[label] == 1]

        if positive_labels:
            title_text = "\n".join(positive_labels[:3])
            if len(positive_labels) > 3:
                title_text += f"\n(+{len(positive_labels)-3} more)"
        else:
            title_text = "No labels"

        ax.set_title(title_text, fontsize=9)
    else:
        ax.text(0.5, 0.5, title, ha='center', va='center', fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Sample images saved as 'sample_images.png'")

# ============================================
# STEP 2.9: SAVE PROCESSED DATA
# ============================================

print("\n" + "="*70)
print("Step 2.9: Saving processed data...")
print("="*70)

# Save full processed dataset
output_path = '/content/indiana_processed_full.csv'
frontal_df.to_csv(output_path, index=False)
print(f"✓ Full dataset saved: {output_path}")

# Save train split
train_path = '/content/indiana_train.csv'
train_df.to_csv(train_path, index=False)
print(f"✓ Train set saved: {train_path}")

# Save test split
test_path = '/content/indiana_test.csv'
test_df.to_csv(test_path, index=False)
print(f"✓ Test set saved: {test_path}")

# ============================================
# STEP 2.10: SUMMARY
# ============================================

print("\n" + "="*70)
print("✓ STEP 2 COMPLETE - DATA READY FOR TRAINING!")
print("="*70)

print("\n📊 DATASET SUMMARY:")
print(f"  Total frontal images: {len(frontal_df)}")
print(f"  Train images: {len(train_df)}")
print(f"  Test images: {len(test_df)}")
print(f"  Number of labels: {len(CHEXPERT_LABELS)}")
print(f"  Images location: {IMAGES_PATH}")

print("\n📁 SAVED FILES:")
print(f"  • {output_path}")
print(f"  • {train_path}")
print(f"  • {test_path}")

print("\n📈 NEXT STEP:")
print("  Ready for Step 3: Train Neural Network (CNN)")

# Save important variables for next step
print("\n📌 VARIABLES FOR NEXT STEP:")
print(f"IMAGES_PATH = '{IMAGES_PATH}'")
print(f"TRAIN_CSV = '{train_path}'")
print(f"TEST_CSV = '{test_path}'")
print(f"LABEL_COLUMNS = {CHEXPERT_LABELS}")

print("\n✅ You can now proceed to Step 3: Neural Module Training!")

In [ ]:
"""
STEP 3: NEURAL NETWORK TRAINING
Train DenseNet121 CNN for multi-label chest X-ray classification

INSTRUCTIONS:
1. Copy this ENTIRE code into a NEW cell in Colab
2. Make sure you're using GPU: Runtime → Change runtime type → GPU
3. Run it (Shift + Enter)
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION FROM STEP 2
# ============================================

IMAGES_PATH = '/content/indiana_dataset/images/images_normalized'
TRAIN_CSV = '/content/indiana_train.csv'
TEST_CSV = '/content/indiana_test.csv'

LABEL_COLUMNS = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
                 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
                 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
                 'Pleural Other', 'Fracture', 'Support Devices']

print("="*70)
print("STEP 3: NEURAL NETWORK TRAINING")
print("="*70)

# ============================================
# STEP 3.1: CHECK GPU AVAILABILITY
# ============================================

print("\nStep 3.1: Checking GPU availability...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("  ⚠ WARNING: GPU not available. Training will be SLOW!")
    print("  Go to: Runtime → Change runtime type → Select GPU")

# ============================================
# STEP 3.2: CREATE DATASET CLASS
# ============================================

print("\n" + "="*70)
print("Step 3.2: Creating dataset class...")
print("="*70)

class CXRDataset(Dataset):
    """Dataset class for chest X-ray images"""

    def __init__(self, csv_file, images_path, label_columns, transform=None):
        """
        Args:
            csv_file: Path to CSV with image filenames and labels
            images_path: Path to images folder
            label_columns: List of label column names
            transform: Optional image transforms
        """
        self.dataframe = pd.read_csv(csv_file)
        self.images_path = images_path
        self.label_columns = label_columns
        self.transform = transform

        print(f"  Loaded {len(self.dataframe)} samples")

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Get image filename
        filename = self.dataframe.iloc[idx]['filename']
        img_path = os.path.join(self.images_path, filename)

        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            # If image loading fails, return black image
            image = Image.new('RGB', (224, 224), color='black')

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        # Get labels
        labels = torch.FloatTensor([
            self.dataframe.iloc[idx][label]
            for label in self.label_columns
        ])

        return image, labels

# ============================================
# STEP 3.3: DEFINE TRANSFORMS
# ============================================

print("\nStep 3.3: Defining image transforms...")

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("✓ Transforms defined")
print("  Train: Resize, HorizontalFlip, Rotation, ColorJitter, Normalize")
print("  Test: Resize, Normalize")

# ============================================
# STEP 3.4: CREATE DATALOADERS
# ============================================

print("\n" + "="*70)
print("Step 3.4: Creating dataloaders...")
print("="*70)

BATCH_SIZE = 16  # Adjust if you have GPU memory issues
NUM_WORKERS = 2

print(f"Batch size: {BATCH_SIZE}")

# Create datasets
print("\nCreating train dataset...")
train_dataset = CXRDataset(TRAIN_CSV, IMAGES_PATH, LABEL_COLUMNS, train_transform)

print("Creating test dataset...")
test_dataset = CXRDataset(TEST_CSV, IMAGES_PATH, LABEL_COLUMNS, test_transform)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if device.type == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if device.type == 'cuda' else False
)

print(f"\n✓ Dataloaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

# ============================================
# STEP 3.5: BUILD MODEL
# ============================================

print("\n" + "="*70)
print("Step 3.5: Building DenseNet121 model...")
print("="*70)

class CXRClassifier(nn.Module):
    """DenseNet121 for multi-label CXR classification"""

    def __init__(self, num_classes=14, pretrained=True):
        super(CXRClassifier, self).__init__()

        # Load pretrained DenseNet121
        self.densenet = models.densenet121(pretrained=pretrained)

        # Replace classifier
        num_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
            nn.Sigmoid()  # Sigmoid for multi-label
        )

    def forward(self, x):
        return self.densenet(x)

# Initialize model
num_classes = len(LABEL_COLUMNS)
model = CXRClassifier(num_classes=num_classes, pretrained=True)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model initialized:")
print(f"  Architecture: DenseNet121")
print(f"  Output classes: {num_classes}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# ============================================
# STEP 3.6: LOSS, OPTIMIZER, SCHEDULER
# ============================================

print("\n" + "="*70)
print("Step 3.6: Setting up training components...")
print("="*70)

# Loss function for multi-label classification
criterion = nn.BCELoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.5
)

print("✓ Training setup:")
print(f"  Loss: Binary Cross Entropy")
print(f"  Optimizer: Adam (lr=0.0001)")
print(f"  Scheduler: ReduceLROnPlateau")

# ============================================
# STEP 3.7: TRAINING FUNCTIONS
# ============================================

print("\n" + "="*70)
print("Step 3.7: Defining training functions...")
print("="*70)

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    num_batches = len(dataloader)

    pbar = tqdm(dataloader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / num_batches
    return epoch_loss

def validate(model, dataloader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    num_batches = len(dataloader)

    all_outputs = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation', leave=False)
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            all_outputs.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / num_batches

    # Calculate AUC
    all_outputs = np.vstack(all_outputs)
    all_labels = np.vstack(all_labels)

    auc_scores = []
    for i in range(all_labels.shape[1]):
        if len(np.unique(all_labels[:, i])) > 1:
            try:
                auc = roc_auc_score(all_labels[:, i], all_outputs[:, i])
                auc_scores.append(auc)
            except:
                pass

    mean_auc = np.mean(auc_scores) if auc_scores else 0.0

    return epoch_loss, mean_auc, all_outputs, all_labels

print("✓ Training functions ready")

# ============================================
# STEP 3.8: TRAIN THE MODEL
# ============================================

print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)

NUM_EPOCHS = 30  # Start with 5 epochs (you can increase later)
SAVE_PATH = '/content/best_cxr_model.pth'

print(f"\nTraining for {NUM_EPOCHS} epochs...")
print(f"Model will be saved to: {SAVE_PATH}\n")

best_val_loss = float('inf')
history = {
    'train_loss': [],
    'val_loss': [],
    'val_auc': []
}

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")

    # Train
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_auc, val_outputs, val_labels = validate(
        model, test_loader, criterion, device
    )

    # Update scheduler
    scheduler.step(val_loss)

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    # Print results
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val AUC:    {val_auc:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_auc': val_auc,
            'label_columns': LABEL_COLUMNS
        }, SAVE_PATH)
        print(f"  ✓ Best model saved! (Val Loss: {val_loss:.4f})")

print("\n" + "="*70)
print("✓ TRAINING COMPLETE!")
print("="*70)

# ============================================
# STEP 3.9: PLOT TRAINING HISTORY
# ============================================

print("\nStep 3.9: Plotting training history...")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(range(1, NUM_EPOCHS+1), history['train_loss'],
            label='Train Loss', marker='o', linewidth=2)
axes[0].plot(range(1, NUM_EPOCHS+1), history['val_loss'],
            label='Val Loss', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# AUC plot
axes[1].plot(range(1, NUM_EPOCHS+1), history['val_auc'],
            label='Val AUC', marker='o', color='green', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('AUC Score', fontsize=12)
axes[1].set_title('Validation AUC Score', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training history saved as 'training_history.png'")

# ============================================
# STEP 3.10: EVALUATE BEST MODEL
# ============================================

print("\n" + "="*70)
print("Step 3.10: Evaluating best model...")
print("="*70)

# Load best model
checkpoint = torch.load(SAVE_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✓ Loaded best model:")
print(f"  Epoch: {checkpoint['epoch']+1}")
print(f"  Val Loss: {checkpoint['val_loss']:.4f}")
print(f"  Val AUC: {checkpoint['val_auc']:.4f}")

# Evaluate
_, _, test_outputs, test_labels = validate(model, test_loader, criterion, device)

# Per-class performance
print("\n" + "="*70)
print("PER-CLASS PERFORMANCE")
print("="*70)
print(f"{'Label':<30s} {'AUC':>8s} {'Count':>8s}")
print("-" * 70)

for i, label_name in enumerate(LABEL_COLUMNS):
    if len(np.unique(test_labels[:, i])) > 1:
        try:
            auc = roc_auc_score(test_labels[:, i], test_outputs[:, i])
            count = int(test_labels[:, i].sum())
            print(f"{label_name:<30s} {auc:>8.3f} {count:>8d}")
        except:
            print(f"{label_name:<30s} {'N/A':>8s} {int(test_labels[:, i].sum()):>8d}")

# Overall mean AUC
valid_aucs = []
for i in range(test_labels.shape[1]):
    if len(np.unique(test_labels[:, i])) > 1:
        try:
            valid_aucs.append(roc_auc_score(test_labels[:, i], test_outputs[:, i]))
        except:
            pass

mean_auc = np.mean(valid_aucs) if valid_aucs else 0.0

print("-" * 70)
print(f"{'MEAN AUC':<30s} {mean_auc:>8.3f}")
print("=" * 70)

# ============================================
# STEP 3.11: SAVE PREDICTIONS
# ============================================

print("\n" + "="*70)
print("Step 3.11: Saving predictions...")
print("="*70)

# Load test CSV
test_df = pd.read_csv(TEST_CSV)

# Add prediction columns
for i, label in enumerate(LABEL_COLUMNS):
    test_df[f'{label}_prob'] = test_outputs[:, i]

# Save
predictions_path = '/content/test_predictions.csv'
test_df.to_csv(predictions_path, index=False)

print(f"✓ Predictions saved: {predictions_path}")

# Also save model to Google Drive
drive_model_path = '/content/drive/MyDrive/cxr_best_model.pth'
try:
    torch.save(checkpoint, drive_model_path)
    print(f"✓ Model saved to Google Drive: {drive_model_path}")
except:
    print("⚠ Could not save to Google Drive")

# ============================================
# STEP 3.12: SUMMARY
# ============================================

print("\n" + "="*70)
print("✓ STEP 3 COMPLETE - NEURAL MODULE TRAINED!")
print("="*70)

print(f"\n📊 TRAINING SUMMARY:")
print(f"  Total epochs: {NUM_EPOCHS}")
print(f"  Best Val Loss: {best_val_loss:.4f}")
print(f"  Final Mean AUC: {mean_auc:.4f}")
print(f"  Model architecture: DenseNet121")

print(f"\n📁 SAVED FILES:")
print(f"  • Model: {SAVE_PATH}")
print(f"  • Predictions: {predictions_path}")
print(f"  • Training plot: training_history.png")

print(f"\n📈 NEXT STEP:")
print(f"  Ready for Step 4: Fuzzy Logic & Symbolic Reasoning")

print(f"\n✅ Neural module complete! Ready for fuzzy logic integration!")